# Fine-Tuning MiniLM for Medical Retrieval

Authors: Aswini Sivakumar and Vishwa Gosavi

This notebook shows how to fine-tune a **SentenceTransformer MiniLM embedding model** on **PubMedQA** so it can produce higher-quality embeddings for biomedical retrieval tasks.

## Project goal
The aim is to improve retrieval performance for medical question-answering workflows by adapting a general-purpose embedding model to the biomedical domain using contrastive learning.

## The workflow in this notebook is:
- installing dependencies
- importing libraries and setting reproducibility
- preparing PubMedQA training and evaluation data
- building an information retrieval evaluation set
- measuring baseline retrieval performance
- fine-tuning the embedding model with contrastive loss
- saving the trained model
- evaluating retrieval performance after fine-tuning
- generating an example embedding for downstream use

## 1. Install dependencies

This step installs the libraries required for the project:
- **sentence-transformers** for embedding models and training
- **datasets** for loading PubMedQA
- **accelerate** for efficient training support

In [ ]:
!pip install -q -U sentence-transformers datasets accelerate

## 2. Import libraries

Here we import the core libraries used throughout the notebook.

These packages support:
- tensor operations and model training
- dataset loading and preprocessing
- sentence embedding fine-tuning
- retrieval-based evaluation

In [ ]:
import torch
import random
import numpy as np
from datasets import load_dataset
from sentence_transformers import (
    SentenceTransformer,
    SentenceTransformerTrainer,
    SentenceTransformerTrainingArguments,
    losses
)
from sentence_transformers.evaluation import InformationRetrievalEvaluator

## 3. Configure experiment settings

This section defines the main training configuration, such as:
- base embedding model
- sequence length
- batch size
- learning rate
- number of epochs

In [ ]:
# ---------------------------------------------------------
# Setup
# ---------------------------------------------------------

#MODEL_ID = "BAAI/bge-small-en-v1.5"
#MAX_SEQ_LENGTH = 384
MODEL_ID = "sentence-transformers/all-MiniLM-L12-v2"
MAX_SEQ_LENGTH = 256 # MiniLM is natively optimized for 256 tokens
BATCH_SIZE = 32 # Fits easily on Colab T4 for a small model
EPOCHS = 2

## 4. Prepare the PubMedQA dataset

In this section, the notebook loads:
- **PubMedQA unlabeled split** for contrastive training
- **PubMedQA labeled split** for retrieval evaluation

The training data is converted into **anchor-positive pairs**, where:
- the **question** acts as the anchor
- the **context** acts as the positive passage

This setup teaches the embedding model to place medically related questions and supporting passages closer together in vector space.

In [ ]:
# ---------------------------------------------------------
# Dataset Preparation
# ---------------------------------------------------------
print("Loading datasets...")
# Load unlabeled data for training and labeled data for evaluation
train_dataset_raw = load_dataset("qiaojin/PubMedQA", "pqa_unlabeled", split="train")
eval_dataset_raw = load_dataset("qiaojin/PubMedQA", "pqa_labeled", split="train")

# Shuffle and sample 10,000 records
train_dataset_raw = train_dataset_raw.shuffle(seed=42).select(range(10000))

# Preprocess training data into pairs: {"anchor": question, "positive": context}
def format_train_data(example):
    return {
        "anchor": example["question"],
        "positive": " ".join(example["context"]["contexts"])
    }

train_dataset = train_dataset_raw.map(
    format_train_data,
    remove_columns=train_dataset_raw.column_names # Remove raw columns
)

## 5. Build the retrieval evaluation pipeline

To measure retrieval quality, we construct three standard components:
- a **corpus** of candidate documents
- a set of **queries**
- a **relevance mapping** that tells us which documents are correct for each query

This allows the model to be evaluated as a retrieval system, not just as a text encoder. The evaluation step helps quantify whether fine-tuning improves how well the model matches medical questions with relevant evidence.

In [ ]:
# ---------------------------------------------------------
# Evaluation Pipeline Setup
# ---------------------------------------------------------
print("Building Evaluation Corpus...")
corpus = {}          # doc_id -> document_text
queries = {}         # query_id -> query_text
relevant_docs = {}   # query_id -> set([doc_id])

context_to_id = {}
doc_idx = 0

# Extract queries and construct unique corpus from the eval split
for i, row in enumerate(eval_dataset_raw):
    q_id = f"q_{i}"
    queries[q_id] = row["question"]

    # Concatenate context
    context_str = " ".join(row["context"]["contexts"])

    # Avoid duplicate contexts in corpus
    if context_str not in context_to_id:
        doc_id = f"d_{doc_idx}"
        corpus[doc_id] = context_str
        context_to_id[context_str] = doc_id
        doc_idx += 1
    else:
        doc_id = context_to_id[context_str]

    relevant_docs[q_id] = set([doc_id])

evaluator = InformationRetrievalEvaluator(
    queries=queries,
    corpus=corpus,
    relevant_docs=relevant_docs,
    name="pubmedqa-eval",
    mrr_at_k=[10],
    precision_recall_at_k=[5, 10], # Extracts Recall@5 and Recall@10
    show_progress_bar=True
)

## 6. Load the base model and establish a baseline

Before training, we evaluate the original pre-trained MiniLM model on the retrieval benchmark.

This gives us a **baseline** so we can compare performance before and after fine-tuning.

In [ ]:
# ---------------------------------------------------------
# Model Loading & Base Evaluation
# ---------------------------------------------------------
print(f"Loading base model: {MODEL_ID}")
model = SentenceTransformer(MODEL_ID)
model.max_seq_length = MAX_SEQ_LENGTH

print("\n--- Evaluating Base (Pre-trained) Model ---")
base_metrics = evaluator(model, output_path="./eval_base")
print(f"Base Model Results: {base_metrics}")

## 7. Authenticate with Hugging Face

This step logs into Hugging Face so the notebook can access or upload models if needed.

Use your personal access token when prompted. If you are working in a private environment and do not need to push artifacts, this step may be optional.

In [ ]:
from huggingface_hub import login
login() # This will prompt you to paste your token

## 8. Define the training objective

This section sets up the fine-tuning process.

The notebook uses **Multiple Negatives Ranking Loss**, a common contrastive objective for retrieval models. The idea is:
- each question should become close to its matching medical context
- other contexts in the batch serve as implicit negatives


In [ ]:
# ---------------------------------------------------------
# Training Objective & Setup
# ---------------------------------------------------------
# InfoNCE Loss: Uses anchor + positive pairs, and treats all other positives in the batch as negatives
train_loss = losses.MultipleNegativesRankingLoss(model)

# Define Hugging Face Training Arguments
args = SentenceTransformerTrainingArguments(
    output_dir="./minilm-pubmedqa-finetuned",
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    warmup_ratio=0.1,
    learning_rate=2e-5,
    fp16=True, # Enable mixed precision for speed and memory savings
    eval_strategy="epoch",    # Evaluate at the end of each epoch
    save_strategy="epoch",
    logging_steps=50,
    seed=42,
    push_to_hub=True, # <-- Add this line
    hub_model_id="AswiniSivakumar/mini-pubmedqa-finetuned" # <-- Add your HF username
)

trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    loss=train_loss,
    evaluator=evaluator
)

## 9. Run contrastive fine-tuning

This is the main training stage. The trainer updates the embedding model so that relevant medical question-context pairs become more similar in embedding space.


In [ ]:
# ---------------------------------------------------------
# Fine-Tuning
# ---------------------------------------------------------
print("\n--- Starting Contrastive Fine-Tuning ---")
trainer.train()

## 10. Save the model and evaluate the final result

Once training is complete, the fine-tuned model is saved and evaluated again on the retrieval benchmark.

Comparing these final metrics against the baseline helps determine whether the domain adaptation improved retrieval quality.

In [ ]:
# ---------------------------------------------------------
# 7. Save & Final Evaluation
# ---------------------------------------------------------
print("\n--- Training Complete! Saving Model ---")
model.save("./minilm-pubmedqa-finetuned/final")

print("\n--- Evaluating Fine-Tuned Model ---")
finetuned_metrics = evaluator(model, output_path="./eval_finetuned")
print(f"Fine-Tuned Model Results: {finetuned_metrics}")

## 11. Example inference with the fine-tuned model

This final section shows how to load the saved model and generate an embedding for a sample medical query.

This is useful for demonstrating how the model can be reused in downstream applications such as:
- semantic search
- document retrieval
- medical RAG pipelines

In [ ]:
from sentence_transformers import SentenceTransformer

# Load your custom, fine-tuned medical model
model = SentenceTransformer("./minilm-pubmedqa-finetuned/final")

# Generate an embedding for a user query
query_embedding = model.encode("What are the side effects of Lisinopril?")

## 12. Inspect the generated embedding

The output vector shown below is the numerical embedding representation of the sample query.

In practice, this embedding would be compared against stored document embeddings inside a vector database to retrieve the most relevant medical passages.

In [ ]:
print(query_embedding)